# Step 2. Cretae the retriever model

In [1]:
import os
import pickle
from langchain_core.documents import Document

from gensim import corpora
from gensim.parsing import strip_tags, strip_numeric, \
    strip_multiple_whitespaces, stem_text, strip_punctuation, \
    remove_stopwords, preprocess_string, strip_non_alphanum
from gensim import models
from gensim import similarities
import pymorphy3

from tqdm.notebook import tqdm

In [2]:
morph_analyzer = pymorphy3.MorphAnalyzer()

In [3]:
SPLITS_PATH = './splits'
GENSIM_PATH = './gensim'

In [4]:
with open(os.path.join(SPLITS_PATH, 'splits.pkl'), 'rb') as f:
    splits = pickle.load(f)
print(len(splits))

100


### Processing the splitted text

In [5]:
STOP_WORDS = ['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то',
              'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за',
              'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще',
              'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли',
              'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь',
              'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей',
              'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя',
              'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже',
              'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому',
              'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой',
              'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда',
              'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над',
              'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая',
              'много', 'разве', 'три', 'эту', 'моя', 'впрочем', 'хорошо', 'свою', 'этой',
              'перед', 'иногда', 'лучше', 'чуть', 'том', 'нельзя', 'такой', 'им', 'более',
              'всегда', 'конечно', 'всю', 'между']

In [6]:
# Filters to be executed in pipeline
transform_to_lower = lambda s: s.lower()
CLEAN_FILTERS = [
                strip_tags,
                # strip_numeric,
                strip_punctuation,
                strip_non_alphanum,
                strip_multiple_whitespaces,
                transform_to_lower,
                ]

In [7]:
# Method does the filtering of all the unrelevant text elements
def cleaning_pipe(text:str) -> list[str]:
    # Invoking gensim.parsing.preprocess_string method with set of filters
    processed_words = preprocess_string(text, CLEAN_FILTERS)
    processed_words = [s for s in processed_words if len(s) > 1]
    processed_words = [s for s in processed_words if s not in STOP_WORDS]
    processed_words = [morph_analyzer.parse(s)[0].normal_form for s in processed_words]
    return processed_words

In [8]:
morph_analyzer.parse('Шеветты')[0].normal_form

'шеветта'

In [9]:
%%time
processed_docs = []
for doc in tqdm(splits):
    processed_docs.append(cleaning_pipe(doc.page_content))

  0%|          | 0/100 [00:00<?, ?it/s]

CPU times: total: 5.11 s
Wall time: 5.1 s


In [10]:
processed_docs[10][:10]

['громила',
 'дело',
 'проламывать',
 'громила',
 'шонбрунновский',
 'бронировать',
 'ворота',
 'райделла',
 'испытать',
 'мгновенный']

### Create Dictionary, Model, Index

In [11]:
%%time
dictionary = corpora.Dictionary(processed_docs)
with open(os.path.join(GENSIM_PATH, 'dictionary.pkl'), 'wb') as h:
    pickle.dump(dictionary, h)
print(dictionary)

Dictionary<11091 unique tokens: ['1939', '1974', '1977', 'австралийский', 'автомат']...>
CPU times: total: 62.5 ms
Wall time: 76 ms


In [12]:
%%time
bow_corpus = [dictionary.doc2bow(text) for text in processed_docs]
model = models.TfidfModel(bow_corpus)
model.save(os.path.join(GENSIM_PATH, 'tfidf_model.mo'))

CPU times: total: 109 ms
Wall time: 115 ms


In [13]:
%%time
index = similarities.SparseMatrixSimilarity(model[bow_corpus], num_features=len(dictionary))
with open(os.path.join(GENSIM_PATH, 'similarity_index.pkl'), 'wb') as h:
    pickle.dump(index, h)

CPU times: total: 62.5 ms
Wall time: 70.3 ms


### Search engine and test

In [14]:
def get_closest_n(query, n):
    '''get the top matching docs as per cosine similarity
    between tfidf vector of query and all docs'''
    query_document = cleaning_pipe(query)
    query_bow = dictionary.doc2bow(query_document)
    sims = index[model[query_bow]]
    top_idx = sims.argsort()[-1*n:][::-1]
    
    result = []
    for idx in top_idx:
        similarity = round(float(sims[idx]), 3)
        doc = splits[idx]
        result.append((doc, similarity))
    return result

In [15]:
query = """Какой персонаж, работающий на 'Интенсекьюр' уже два года, рассказал Райделлу о существовании домов-невидимок, известных как 'стелсы'?"""

In [16]:
# Check the result
result = get_closest_n(query, n=3)
for doc, similarity in result:
    for k,v in doc.metadata.items():
        if k in ['chapter', 'section', 'size']:
            print(f'{k}: {v}')
    # print(f'Context: {doc.page_content[:500]}...')
    print(f'similarity: {similarity}\n')

chapter: 2. "ГРОМИЛА" В ДЕЛЕ
section: Райделл вставил ключ...
size: 8156
similarity: 0.236

chapter: 26. "ЦВЕТНЫЕ ЛЮДИ"
section: Может в этом мормонском...
size: 11731
similarity: 0.039

chapter: 29. МЕРТВЫЙ МОЛЛ
section: Лавлесс...
size: 10944
similarity: 0.037



In [17]:
splits[10]

Document(metadata={'title': 'Уильям Гибсон. Виртуальный свет', 'chapter': '2. "ГРОМИЛА" В ДЕЛЕ', 'section': 'Проламывая "Громилой"...', 'id': '713a46dedbe4a75dc6918efbe5bd4878', 'size': 734, 'collection': 'Virtual_Light'}, page_content='2. "ГРОМИЛА" В ДЕЛЕ\n\nПроламывая "Громилой" шонбрунновские бронированные ворота, Райделл испытал мгновенное ощущение чего-то очень высокого, очень чистого, клинически пустого - деланье, лишенное всякого думанья, бредовое адреналиновое возбуждение с утратой всех прочих аспектов своего "я".\nИ в это время - когда он боролся с рулем, прорываясь через японский сад камней, когда пробивал тугое бронестекло, рассыпавшееся со второго удара, и осколки падали медленно, как во сне, - в это время он вспомнил, что примерно то же самое ощущал и тогда, выхватывая револьвер, нажимая курок, выплескивая мозги Кеннета Терви на бесконечный, как звездное небо, простор белой, загрунтованной под краску стены, которую так никто никогда и не покрасил.')

In [20]:
query = """Райделл испытал мгновенное ощущение чего-то очень высокого, очень чистого, клинически пустого - деланье, лишенное всякого думанья, бредовое адреналиновое возбуждение с утратой всех прочих аспектов своего я"""

In [21]:
result = get_closest_n(query, n=3)
for doc, similarity in result:
    for k,v in doc.metadata.items():
        if k in ['chapter', 'section', 'size']:
            print(f'{k}: {v}')
    # print(f'Context: {doc.page_content[:500]}...')
    print(f'similarity: {similarity}\n')

chapter: 2. "ГРОМИЛА" В ДЕЛЕ
section: Проламывая "Громилой"...
size: 734
similarity: 0.586

chapter: 28. АР-ВИ
section: вспыхнувшие лампы...
size: 3406
similarity: 0.023

chapter: 17. МЫШЕЛОВКА
section: А тыто пробовал когда...
size: 5190
similarity: 0.02

